# Student Registration 분석

## 1. 데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

In [ ]:
student_re = pd.read_csv("../CSV_files/studentRegistration.csv")
student_re.head()

In [ ]:
student_re.describe()

In [ ]:
student_re.info()

In [ ]:
student_re.isnull().sum()

In [ ]:
cat_cols = ["code_module", "code_presentation", "date_registration", "date_unregistration"]
for c in cat_cols:
    print(f"--- {c} ---")
    print(student_re[c].value_counts(dropna=False), "\n")

- code_module (과목 코드): 어떤 전공 수업인지 나타내는 '과목 이름표'

- code_presentation (개설 학기): 수업이 열린 '연도와 학기 정보'. 

- date_registration (수강 등록일): 학생이 '수강 신청 버튼을 누른 날짜'.  

- date_unregistration (수강 취소일): 학생이 '수강 취소(중도 포기) 버튼을 누른 날짜'.  

우리 프로젝트의 핵심: 이 칸에 숫자가 적혀 있는 학생이 바로 우리가 예측해야 하는 '이탈자'이고, 끝까지 포기하지 않아서 빈칸(NaN 또는 공백)으로 남아있는 학생이 '완수자'

## 1-1. label_churn 정의

이후 분석에서 01_Student_info_EDA.ipynb와 같은 기준으로 이탈 여부를 비교하려면 label_churn 정의를 먼저 통일해둬야 합니다. 

그래서 이 노트북에서도 studentInfo.csv를 불러와 동일한 규칙(Withdrawn=이탈, 나머지=완수)으로 label_churn을 만들어두고, 앞으로의 조인/분석에서
계속 재사용합니다.

In [ ]:
# studentInfo.csv를 불러옵니다. (이 노트북에서 처음 불러오는 것이라, 이후 2번 조인에서도 이 info를 그대로 재사용합니다)
info = pd.read_csv("../CSV_files/studentInfo.csv")

# 01_Student_info_EDA.ipynb와 동일한 기준으로 이탈 여부를 정의합니다.
# 훈련 결과가 'Withdrawn'이면 "이탈", 나머지(Pass/Fail/Distinction)는 "완수"로 분류합니다.
# (조기 이탈률만 확인하기 위해 Fail도 일단 "완수"로 분류)
info["label_churn"] = info["final_result"].apply(lambda x: "이탈" if x == "Withdrawn" else "완수")

# 결과가 어떻게 나뉘었는지 퍼센트로 확인합니다. (01번 노트북과 같은 값(완수 0.688 / 이탈 0.312)이 나와야 정상입니다)
print(info["label_churn"].value_counts(normalize=True).round(3))

## 2. 라벨 정합성 검증 (studentInfo와 조인)

studentInfo.csv에서 `final_result == "Withdrawn"`인 학생과, 이 파일의 `date_unregistration`이 있는 학생은 둘 다 "이탈"을 나타내는 값이므로 원래는 정확히 같은 사람들이어야 합니다.

그런데 위에서 본 것처럼 Withdrawn은 10,156명, date_unregistration이 있는 학생은 10,072명으로 84명 차이가 납니다. 

두 라벨이 정말 같은 대상을 가리키는지 직접 조인해서 확인합니다.

In [ ]:
# 1-1에서 이미 불러온 info(studentInfo.csv)를 그대로 재사용해서,
# (code_module, code_presentation, id_student) 키로 student_re와 조인합니다.
# 이 3개 컬럼을 합치면 "어떤 학생이 어떤 과목/학기를 수강했는지"를 유일하게 식별할 수 있습니다.
merged = pd.merge(
    info, student_re,
    on=["code_module", "code_presentation", "id_student"],
    how="left"  # info 기준으로 붙여서, info의 학생이 하나도 빠지지 않게 합니다.
)

is_withdrawn = merged["final_result"] == "Withdrawn"
has_unreg_date = merged["date_unregistration"].notna()

print("final_result == Withdrawn 인원:", is_withdrawn.sum())
print("date_unregistration 값이 있는 인원:", has_unreg_date.sum())

# Case A: "Withdrawn"이라고 표시됐는데 이탈 날짜 기록이 없는 학생
case_a = merged[is_withdrawn & ~has_unreg_date]
# Case B: Withdrawn이 아닌데 이탈 날짜가 기록된 학생
case_b = merged[~is_withdrawn & has_unreg_date]

print(f"\nWithdrawn인데 date_unregistration이 없는 경우: {len(case_a)}건")
print(case_a["final_result"].value_counts())

print(f"\nWithdrawn이 아닌데 date_unregistration이 있는 경우: {len(case_b)}건")
print(case_b["final_result"].value_counts())

# [검산] 10,156(Withdrawn) - 93(case_a) + 9(case_b) = 10,072 -> 처음 봤던 84명 차이가 이렇게 설명됩니다.

**[발견]**
- **93명**: `final_result="Withdrawn"`인데 `date_unregistration`이 비어 있습니다. 이탈 처리는 됐지만
  정확한 이탈 시점이 기록에서 누락된 케이스로 보입니다. 이 학생들은 "언제 이탈했는지"를 알 수 없으므로
  이후 이탈 시점 분석에서는 자동으로 제외됩니다. (참고로 이 93명은 date_registration은 정상적으로
  있어서, 4-1에서 다룬 "등록일 자체가 없는 39명"과는 다른 그룹입니다.)
- **9명**: `final_result="Fail"`인데 `date_unregistration`이 있습니다. 즉 취소 기록은 남아있지만
  최종적으로는 이탈(Withdrawn)이 아니라 낙제(Fail)로 처리된 경우입니다. (아래 2-2에서 상세 확인)
- 10,156 - 93 + 9 = 10,072 로 정확히 맞아떨어져서, 처음 봤던 84명 차이의 원인이 이 두 그룹임을 확인했습니다.

**[결론]** `label_churn`(final_result 기반)과 `date_unregistration`(날짜 기반)은 100% 일치하지 않습니다.
앞으로 "이탈 시점"을 분석할 때는 date_unregistration이 있는 10,072명을 기준으로 진행하되,
"이탈 여부" 자체를 판단할 때는 지금까지 써온 studentInfo의 final_result / label_churn 기준을 유지하는 게 안전합니다.

## 2-1. 조인 안전성 및 날짜 논리 검증

2번의 결론을 그대로 믿기 전에, 조인 자체와 날짜 값에 문제가 없는지 먼저 확인해야 합니다.
1) student_re에 (code_module, code_presentation, id_student) 키 중복이 있으면 merge할 때
행이 불어나서(fan-out) 지금까지 계산한 모든 인원수 집계가 틀어집니다.
2) date_unregistration이 date_registration보다 빠르다면 "등록하기도 전에 이탈"한 셈이라
논리적으로 말이 안 되는 데이터 오류입니다. 두 가지 다 미리 점검해둡니다.

In [ ]:
# 1) student_re의 키(code_module, code_presentation, id_student) 중복 여부 확인
dup_keys = student_re.duplicated(subset=["code_module", "code_presentation", "id_student"]).sum()
print("student_re 키 중복 행 수:", dup_keys)
print("info 원본 행 수:", info.shape[0], "/ merge 후 행 수:", merged.shape[0])
# 두 값이 같아야 merge가 1:1로 안전하게 이루어졌다는 뜻입니다 (행이 늘거나 줄지 않음).

# 2) 등록일보다 이탈일이 더 빠른(논리적으로 불가능한) 케이스 확인
both = merged.dropna(subset=["date_registration", "date_unregistration"])
bad = both[both["date_unregistration"] < both["date_registration"]]
print("\ndate_unregistration < date_registration 인 이상 케이스 수:", len(bad))

**[확인 결과]** 키 중복 0건, merge 전후 행 수 동일(32,593건) → 조인이 1:1로 안전하게 이뤄졌습니다.
날짜 역전 케이스도 0건 → date_unregistration 값들은 논리적으로 문제가 없어 그대로 신뢰하고 써도 됩니다.

## 2-2. case_b(9명) 상세 확인 — 처음 추측이 틀렸던 부분

2번에서 "Fail인데 date_unregistration이 있는 9명은 학기 막바지에 취소해서 이미 성적 처리가
됐을 것"이라고 추측했었는데, 실제 값을 찍어보고 그 추측이 맞는지 검증합니다.

In [ ]:
# courses.csv(강좌 공식 길이)와 조인해서, 9명의 취소 시점이 강좌 막바지였는지 직접 확인합니다.
courses = pd.read_csv("../CSV_files/courses.csv")
merged_len = pd.merge(merged, courses, on=["code_module", "code_presentation"], how="left")

case_b_len = merged_len[(~is_withdrawn) & has_unreg_date]
cols = ["code_module", "code_presentation", "id_student",
        "date_registration", "date_unregistration", "module_presentation_length"]
print(case_b_len[cols].to_string())

# 강좌 길이의 80% 지점 이후(=막바지)에 취소한 경우가 몇 명인지 확인합니다.
late_cancel = (case_b_len["date_unregistration"] >= case_b_len["module_presentation_length"] * 0.8).sum()
print(f"\n강좌 막바지(80% 지점 이후)에 취소한 경우: {late_cancel}명 / {len(case_b_len)}명")

**[정정]** 처음 추측과 달리, 9명 중 **8명은 취소 시점이 0일 근처(강좌 시작 직전~직후)로 매우 이릅니다**
(0일이 6명, -4일·-7일이 각 1명). 나머지 1명(166일)이 상대적으로 가장 늦지만, 이마저도 강좌 길이의
80% 지점(약 210일)에는 못 미쳐 "막바지 취소"로 보기는 어렵습니다(80% 이후 취소 0명/9명).

즉 이 학생들은 대부분 강좌 초반에 취소 기록을 남겼는데도 최종 성적은 Withdrawn이 아니라 Fail로
처리된, 설명이 애매한 데이터 불일치 케이스입니다. 취소 후 재등록했거나 행정적으로 취소가
철회됐을 가능성이 있지만, 이 파일만으로는 정확한 이유를 확정할 수 없습니다. 표본이 9명으로
매우 적어 전체 분석에 미치는 영향은 미미하지만, date_unregistration을 "이탈 여부"의 직접적인
대체 지표로 쓰면 안 된다는 근거가 하나 더 늘었습니다.

## 3. 이탈 시점(date_unregistration) 분포 분석

01번 노트북에서는 "이탈했는지 여부"만 알 수 있었고, "언제" 이탈했는지는 알 수 없었습니다.
이 프로젝트의 목표가 "조기 이탈률 개선"인 만큼, date_unregistration으로 실제 이탈 타이밍을
확인하는 게 이 파일의 핵심 가치입니다. date_unregistration은 강좌 시작일을 0일로 기준삼은
값이라, 음수면 강좌가 시작되기도 전에 이미 취소했다는 뜻입니다.

In [ ]:
# 이탈 기록이 있는 학생만 대상으로 분포를 봅니다 (date_unregistration이 NaN인 학생은 이탈 안 한 학생).
plt.figure(figsize=(10, 5))
sns.histplot(merged["date_unregistration"].dropna(), bins=50, kde=True)

# 0일 = 강좌 시작일. 이 선을 기준으로 왼쪽(음수)은 "시작도 못 해보고 이탈"한 학생입니다.
plt.axvline(0, color="red", linestyle="--", label="강좌 시작일(0일)")
plt.title("이탈 시점(date_unregistration) 분포")
plt.xlabel("강좌 시작일 기준 경과일 (음수=시작 전)")
plt.ylabel("학생 수")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 이탈 시점을 3개 구간으로 나눠서 각 구간에 몇 명이 있는지 세어봅니다.
unreg = merged["date_unregistration"].dropna()

def bucket(x):
    if x < 0:
        return "강좌 시작 전 이탈"      # 등록만 하고 강좌가 시작되기도 전에 취소
    elif x <= 30:
        return "초반 조기이탈(0~30일)"   # 강좌 시작 후 한 달 이내 이탈
    else:
        return "중후반 이탈(30일 초과)"  # 그 이후 이탈

order = ["강좌 시작 전 이탈", "초반 조기이탈(0~30일)", "중후반 이탈(30일 초과)"]
bucket_counts = unreg.apply(bucket).value_counts().reindex(order)
bucket_pct = (bucket_counts / len(unreg) * 100).round(1)

print(bucket_counts)
print()
print(bucket_pct.astype(str) + "%")

plt.figure(figsize=(6, 5))
bucket_counts.plot(kind="bar", color=["gray", "salmon", "steelblue"])
plt.title("이탈 시점 구간별 인원")
plt.ylabel("인원 수")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# [해석 포인트] "강좌 시작 전 이탈" + "초반 조기이탈"을 합치면 전체 이탈자의 절반(50.9%)입니다.
# 즉 이탈 원인의 개입 시점을 잡을 때, 강좌 중후반보다 등록 직후~시작 시점에 집중하는 게
# 훨씬 더 많은 이탈을 예방할 수 있다는 뜻입니다.

In [ ]:
# [추가 점검] 이탈 시점의 최댓값(444일)처럼 유난히 큰 값이, 강좌가 공식적으로 끝난 뒤에도
# 이탈 처리된 이상치인지 courses.csv(과목-학기별 공식 길이)와 조인해서 확인합니다.
after_end = merged_len.dropna(subset=["date_unregistration"])
after_end_cnt = (after_end["date_unregistration"] > after_end["module_presentation_length"]).sum()
print(f"강좌 공식 길이보다 늦게 이탈 처리된 경우: {after_end_cnt}건 / 전체 {len(after_end)}건")

**[결론]** 10,072건 중 단 1건만 강좌 공식 길이를 초과해서 이탈 처리됐습니다. 사실상 무시할 수 있는
수준이라, 이탈 시점 값 자체는 별도의 이상치 제거 없이 그대로 사용해도 안전합니다.

## 4. 등록 시점(date_registration)과 이탈의 관계

**가설**: 강좌가 시작된 뒤에 뒤늦게 등록한 학생일수록 진도를 따라가기 어려워 이탈률이 더 높을 것이다.
이 가설이 실제 데이터에서도 맞는지 확인합니다.

In [ ]:
# 완수/이탈 그룹별로 date_registration의 평균, 중앙값을 비교합니다.
print(merged.groupby("label_churn")["date_registration"].describe()[["count", "mean", "50%", "std"]])

plt.figure(figsize=(7, 5))
sns.boxplot(data=merged, x="label_churn", y="date_registration")
plt.title("완수/이탈 그룹별 등록 시점(date_registration) 분포")
plt.xlabel("이탈 여부")
plt.ylabel("강좌 시작일 기준 등록 시점 (음수=시작 전 등록)")
plt.tight_layout()
plt.show()

# [해석 포인트] 이탈 그룹의 평균/중앙값이 완수 그룹보다 오히려 더 마이너스(더 이른 등록)입니다.
# 즉 가설과 반대로, "늦게 등록해서 이탈하는" 패턴이 아니라 "일찍 등록한 사람 중 이탈이 더 많은" 경향이 보입니다.

In [ ]:
# 등록 시점을 구간으로 나눠서 구간별 이탈률(%)을 직접 계산합니다.
def reg_bucket(x):
    if pd.isna(x):
        return "결측"
    elif x < -30:
        return "매우 이른 등록"          # 강좌 시작 한 달도 더 전에 등록
    elif x < 0:
        return "이른 등록(0~30일 전)"
    elif x == 0:
        return "당일 등록"
    else:
        return "늦은 등록(시작 후)"       # 강좌가 시작된 뒤에 등록

merged["reg_bucket"] = merged["date_registration"].apply(reg_bucket)
order = ["매우 이른 등록", "이른 등록(0~30일 전)", "당일 등록", "늦은 등록(시작 후)", "결측"]

rate = merged.groupby("reg_bucket")["label_churn"].value_counts(normalize=True).unstack().reindex(order) * 100
cnt = merged["reg_bucket"].value_counts().reindex(order)
summary = pd.DataFrame({"인원수": cnt, "이탈률(%)": rate["이탈"].round(1)})
print(summary)

plt.figure(figsize=(8, 5))
summary["이탈률(%)"].plot(kind="bar", color="salmon")
plt.title("등록 시점 구간별 이탈률(%)")
plt.ylabel("이탈률(%)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# 참고: date_registration과 이탈 여부(0/1)의 상관계수도 함께 확인합니다.
merged["churn01"] = merged["label_churn"].map({"완수": 0, "이탈": 1})
corr = merged[["date_registration", "churn01"]].corr().iloc[0, 1]
print("\ndate_registration과 이탈(0/1) 상관계수:", round(corr, 3))

In [ ]:
# date_registration이 결측인 45건은 어떤 학생들인지 별도로 확인합니다.
missing_reg = merged[merged["date_registration"].isna()]
print("date_registration 결측 45건의 final_result 분포:")
print(missing_reg["final_result"].value_counts())
print()
print("이 중 Withdrawn 비율:", round((missing_reg["final_result"] == "Withdrawn").mean() * 100, 1), "%")

**[발견 — 가설과 반대]**
- 이탈 그룹의 등록 시점 평균(-78.4일)이 완수 그룹(-65.3일)보다 더 이릅니다.
- 구간별 이탈률도 "매우 이른 등록"이 33.2%로 가장 높고, "늦은 등록(시작 후)"이 오히려 20.9%로 가장 낮습니다.
- 상관계수도 -0.123으로 (약하지만) 음의 상관 — 즉 늦게 등록할수록 이탈률이 낮아지는 방향입니다.
- date_registration이 **결측인 45명 중 39명(86.7%)이 Withdrawn**입니다. 등록일 자체가 기록되지
  않았다는 건, 정식 등록 절차를 마치기도 전에 이미 이탈했을 가능성이 높다는 뜻입니다.

**[결론]** "늦게 등록하면 적응이 어려워 이탈이 많다"는 처음 가설은 데이터와 맞지 않습니다.
오히려 아주 일찍(한 달 이상 전) 등록해두고 정작 강좌가 시작되자 흥미를 잃거나 준비가 부족해
이탈하는 패턴에 가깝습니다. 그리고 등록일 자체가 없는 45명은 "조기 이탈"의 가장 극단적인
형태(등록 완료 전 이탈)로 별도 관리가 필요한 그룹입니다.

## 4-1. date_registration 결측 39명 상세 확인

45명 중 39명이 Withdrawn이라는 걸 확인했는데, 이게 데이터 오류인지 아니면 실제로
"등록 절차를 마치기도 전에 이탈"한 현상인지 직접 들여다봅니다.

In [ ]:
# 39명의 상세 정보를 확인합니다 (어떤 과목/학기이고, 이탈 시점과 인구통계는 어떤지).
withdrawn_39 = missing_reg[missing_reg["final_result"] == "Withdrawn"]

cols = ["code_module", "code_presentation", "id_student", "date_unregistration",
        "imd_band", "disability", "num_of_prev_attempts", "studied_credits"]
print(withdrawn_39[cols].to_string())

In [ ]:
# 1) 이 39명의 date_unregistration(이탈 시점)이 강좌 시작 전(음수)인 비율을 확인합니다.
neg_ratio = (withdrawn_39["date_unregistration"] < 0).mean() * 100
print(f"39명 중 date_unregistration < 0 (강좌 시작 전 이탈): {(withdrawn_39['date_unregistration'] < 0).sum()}명 "
      f"({neg_ratio:.1f}%)")

# 2) 특정 과목에만 몰려있는 현상인지 code_module 분포를 확인합니다.
print("\ncode_module 분포:")
print(withdrawn_39["code_module"].value_counts())

# 3) 같은 id_student가 여러 번(=여러 과목) 이 리스트에 등장하는지 확인합니다.
#    (한 학생이 동시에 여러 과목에서 똑같이 등록일 결측 + 이탈 패턴을 보인다면, 개인 사정일 가능성이 큽니다)
dup_students = missing_reg["id_student"].value_counts()
dup_students = dup_students[dup_students > 1]
print("\n45명 중 여러 과목에 걸쳐 중복 등장하는 학생:")
print(dup_students)

if len(dup_students) > 0:
    sample_id = dup_students.index[0]
    print(f"\n예시 - id_student={sample_id}의 전체 수강 기록:")
    print(merged[merged["id_student"] == sample_id][
        ["code_module", "code_presentation", "date_registration", "date_unregistration", "final_result"]
    ])

**[확인 결과]**
- 39명 중 **35명(89.7%)**이 `date_unregistration`도 음수입니다. 즉 강좌가 시작되기도 전에
  이미 취소한 경우가 대부분입니다. 정식 등록이 시스템에 확정되기 전에 취소해버려서
  `date_registration` 자체가 기록되지 않은 것으로 보이며, 데이터 오류라기보다는
  "매우 이른 이탈"을 나타내는 특성으로 해석하는 게 합리적입니다.
- `code_module`은 BBB(8) · CCC(7) · DDD(12) · EEE(2) · FFF(10)에 걸쳐 나타나고 AAA·GGG에는
  없습니다. 특정 과목 하나에 쏠린 현상이 아니라 여러 과목에서 공통적으로 나타나는 패턴입니다.
- id_student 604480, 2710343은 각각 두 개 과목에서 동시에 이 패턴을 보였고, 604480은 두 과목의
  `date_unregistration`이 정확히 같은 날(-168일)이었습니다. 두 과목을 따로따로 그만둔 게 아니라
  개인 사정으로 한 번에 모든 수강을 접었을 가능성이 높습니다.

**[결론]** 이 39명은 데이터 오류가 아니라 "등록 확정 전 이탈"이라는 실제 현상으로 보입니다.
등록 시점 기반 피처를 만들 때는 이 그룹을 결측으로 그냥 두기보다 "미등록 이탈"이라는 별도
카테고리로 표시해두면, 이후 모델링에서 이 초고위험군을 놓치지 않을 수 있습니다.

## 5. code_module/code_presentation별 평균 이탈 시점 분석

01번 노트북에서는 code_module이 이탈 "여부"를 가장 강하게 좌우하는 변수라는 걸 확인했습니다
(Cramér's V 1위). 여기서는 같은 이탈이라도 "이탈률"이 아니라 "이탈 속도" 관점에서 과목별로
비교합니다. 이탈률이 높다고 해서 반드시 빨리 이탈하는 것은 아닐 수 있기 때문입니다.

In [ ]:
# 과목별 평균 이탈 시점을 계산합니다. (값이 작을수록 더 빨리 이탈한다는 뜻)
avg_by_module = merged.dropna(subset=["date_unregistration"]).groupby("code_module")["date_unregistration"] \
    .agg(["mean", "median", "count"]).sort_values("mean")
print(avg_by_module.round(1))

plt.figure(figsize=(8, 5))
avg_by_module["mean"].plot(kind="bar", color="steelblue")
plt.title("과목(code_module)별 평균 이탈 시점")
plt.ylabel("평균 이탈 시점 (강좌 시작일 기준 경과일)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# "이탈률" 순위와 "이탈 속도" 순위를 나란히 비교합니다. (01번 노트북에서 본 이탈률과 여기서 본 이탈 시점)
rate_by_module = merged.groupby("code_module")["label_churn"].apply(lambda x: (x == "이탈").mean() * 100)

combo = pd.DataFrame({
    "이탈률(%)": rate_by_module,
    "평균이탈시점(일)": avg_by_module["mean"]
}).round(1)
combo["이탈률_순위"] = combo["이탈률(%)"].rank(ascending=False).astype(int)   # 1위 = 이탈률이 가장 높은 과목
combo["이탈속도_순위"] = combo["평균이탈시점(일)"].rank(ascending=True).astype(int)  # 1위 = 가장 빨리 이탈하는 과목

print(combo.sort_values("이탈률_순위"))

# [해석 포인트] CCC는 이탈률 1위지만 이탈속도로는 3위이고, BBB는 이탈률 4위인데 이탈속도는 1위(가장 빠름)입니다.
# 즉 "이탈이 많은 과목"과 "이탈이 빠른 과목"이 서로 다릅니다.
# CCC/DDD는 학기 내내 꾸준히 이탈자가 쌓이는 구조적 문제로, BBB는 초반에 집중적으로 이탈하는
# 문제로 접근해야 할 가능성이 높다는 뜻입니다.

In [ ]:
# 같은 과목이라도 학기(code_presentation)에 따라 평균 이탈 시점이 얼마나 다른지 히트맵으로 확인합니다.
pivot = merged.dropna(subset=["date_unregistration"]).groupby(["code_module", "code_presentation"])["date_unregistration"] \
    .mean().unstack()

plt.figure(figsize=(9, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("과목 x 학기별 평균 이탈 시점 (일)")
plt.xlabel("학기")
plt.ylabel("과목")
plt.tight_layout()
plt.show()

# [해석 포인트] BBB는 학기별로 평균 이탈 시점이 17.7일~56.0일로 매우 큰 편차를 보입니다.
# 특히 2014B 학기는 유독 빠르게(17.7일) 이탈하는데, 이는 과목 자체의 문제라기보다
# 특정 학기의 운영 이슈(강사, 커리큘럼 변경 등)일 가능성이 있어 별도로 들여다볼 필요가 있습니다.

**[결론]** 이탈 "여부"(01번 노트북)와 이탈 "속도"(이 노트북)는 다른 이야기입니다.
- CCC/DDD: 이탈률은 높지만(1·2위) 이탈 속도는 중간(3·5위) — 학기 내내 꾸준히 이탈자가 쌓이는
  구조적 문제로, 커리큘럼/난이도 개편이 필요합니다.
- BBB: 이탈률은 중간(4위)이지만 이탈 속도는 가장 빠릅니다(1위, 평균 33.8일) — 초반 집중 관리로
  많은 이탈을 예방할 수 있는 과목입니다. 특히 2014B 학기는 유독 더 빨라서(17.7일) 그 학기만의
  운영 이슈가 있었는지 별도 확인이 필요합니다.
- AAA: 이탈률도 가장 낮고(6위) 이탈 시점도 가장 늦습니다(101.3일) — 상대적으로 안전한 과목입니다.

고용노동부 훈련과정에 적용한다면, 과목마다 "언제" 개입해야 하는지가 다르다는 뜻이므로
일괄적인 중도이탈 방지책보다 과목별 맞춤 개입 시점 설계가 더 효과적일 수 있습니다.